# PS S6E4 - GBDT CATNUM top4 minimum

目的:
- high-IV OVR scan の top4 categorical columns を起点に、
  単カテゴリ CATNUM を最小構成で追加する
- まずは CatBoost で効くかを確認する
- pair CATNUM / TE / group interaction はまだやらない

対象カテゴリ:
- Crop_Growth_Stage
- Water_Source
- Crop_Type
- Irrigation_Type

## 1. Imports / config

In [ ]:
import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
class CFG:
    COMP_NAME = "playground-series-s6e4"
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 42
    N_SPLITS = 5

    EXP_NAME = "catboost_catnum_top4_min"
    SAVE_DIR = Path("/kaggle/working/exp_20260406_013_gbdt_catnum_top4_min")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    HIGH_IV_TOP4 = [
        "Crop_Growth_Stage",
        "Water_Source",
        "Crop_Type",
        "Irrigation_Type",
    ]

    NUM_COLS = [
        "Soil_pH",
        "Soil_Moisture",
        "Organic_Carbon",
        "Electrical_Conductivity",
        "Temperature_C",
        "Humidity",
        "Rainfall_mm",
        "Sunlight_Hours",
        "Wind_Speed_kmh",
        "Field_Area_hectare",
        "Previous_Irrigation_mm",
    ]

    BASE_CAT_COLS = [
        "Soil_Type",
        "Crop_Type",
        "Crop_Growth_Stage",
        "Season",
        "Irrigation_Type",
        "Water_Source",
        "Region",
        "Mulching_Used",
    ]

    CATNUM_STATS = ["mean", "std", "median", "min", "max"]
    FILLNA_VALUE = -9999.0

    # CFG から削除
    # CLASS_ORDER = ["Low", "Medium", "High"]

    CATBOOST_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1",   # 学習監視用。最終判定は balanced_accuracy
        "iterations": 3000,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 5.0,
        "random_strength": 1.0,
        "subsample": 0.9,
        "bootstrap_type": "Bernoulli",
        "random_seed": SEED,
        "verbose": 200,
        "allow_writing_files": False,
        "task_type": "CPU",
    }


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(CFG.SEED)
print(CFG.SAVE_DIR)

## 2. Load data

In [ ]:
INPUT_DIR = Path(f"/kaggle/input/competitions/{CFG.COMP_NAME}")

train = pd.read_csv(INPUT_DIR / "train.csv")
test = pd.read_csv(INPUT_DIR / "test.csv")
# 使っていないので削除
# original = pd.read_csv("/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv")

print(train.shape, test.shape)
display(train.head())

## 3. Basic checks

In [ ]:
print("train target unique:", train[CFG.TARGET_COL].value_counts(dropna=False).to_dict())
print("top4 high-IV cols:", CFG.HIGH_IV_TOP4)

## 4. CATNUM feature builder

In [ ]:
def build_catnum_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cat_cols: list[str],
    num_cols: list[str],
    stats: list[str],
    target_col: str,
    id_col: str,
    fillna_value: float = -9999.0,
):
    """
    train + test をまとめて group stats を作るのは leakage ではない。
    target を使わない純粋な test-time available feature だから。
    ただし元 train/test 分布差は乗るので、今回は最小検証として採用。
    """
    train_x = train_df.drop(columns=[target_col]).copy()
    test_x = test_df.copy()

    whole = pd.concat(
        [
            train_x.assign(__source="train"),
            test_x.assign(__source="test"),
        ],
        axis=0,
        ignore_index=True,
    )

    created_cols = []

    for cat in cat_cols:
        grp = whole.groupby(cat, dropna=False)

        agg_df = grp[num_cols].agg(stats)
        agg_df.columns = [f"CATNUM__{cat}__{num}__{stat}" for num, stat in agg_df.columns]
        agg_df = agg_df.reset_index()

        whole = whole.merge(agg_df, on=cat, how="left")
        created_cols.extend([c for c in agg_df.columns if c != cat])

    whole[created_cols] = whole[created_cols].fillna(fillna_value)

    train_feat = whole.loc[whole["__source"] == "train"].drop(columns=["__source"]).reset_index(drop=True)
    test_feat = whole.loc[whole["__source"] == "test"].drop(columns=["__source"]).reset_index(drop=True)

    return train_feat, test_feat, created_cols

## 5. Build features

In [ ]:
train_feat, test_feat, catnum_cols = build_catnum_features(
    train_df=train,
    test_df=test,
    cat_cols=CFG.HIGH_IV_TOP4,
    num_cols=CFG.NUM_COLS,
    stats=CFG.CATNUM_STATS,
    target_col=CFG.TARGET_COL,
    id_col=CFG.ID_COL,
    fillna_value=CFG.FILLNA_VALUE,
)

print("n_catnum_cols =", len(catnum_cols))
print(catnum_cols[:10])

display(train_feat.head())

## 6. Feature list

In [ ]:
feature_cols = [c for c in train_feat.columns if c not in [CFG.ID_COL, CFG.TARGET_COL]]
cat_features = [c for c in CFG.BASE_CAT_COLS if c in feature_cols]

print("n_features =", len(feature_cols))
print("n_cat_features =", len(cat_features))
print("n_num_like_features =", len(feature_cols) - len(cat_features))
print("cat_features =", cat_features)

## 7. CV training

In [ ]:
def fit_catboost_cv(train_df, test_df, y, feature_cols, cat_features):
    n_classes = y.nunique()
    oof_proba = np.zeros((len(train_df), n_classes), dtype=np.float32)
    test_proba = np.zeros((len(test_df), n_classes), dtype=np.float32)

    fold_scores = []
    class_names = None

    skf = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED)

    X = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr = X.iloc[tr_idx].copy()
        y_tr = y.iloc[tr_idx].copy()
        X_va = X.iloc[va_idx].copy()
        y_va = y.iloc[va_idx].copy()

        model = CatBoostClassifier(**CFG.CATBOOST_PARAMS)

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_features,
            eval_set=(X_va, y_va),
            use_best_model=True,
        )

        if class_names is None:
            class_names = list(model.classes_)

        va_pred = model.predict(X_va).reshape(-1)
        va_proba = model.predict_proba(X_va)
        te_proba = model.predict_proba(X_test)

        score = balanced_accuracy_score(y_va, va_pred)
        fold_scores.append(score)

        oof_proba[va_idx] = va_proba
        test_proba += te_proba / CFG.N_SPLITS

        print(f"fold {fold} balanced_accuracy = {score:.6f}")

        del model, X_tr, y_tr, X_va, y_va, va_pred, va_proba, te_proba
        gc.collect()

    cv_score = float(np.mean(fold_scores))
    return oof_proba, test_proba, fold_scores, cv_score, class_names

## 8. Run CV

In [ ]:
oof_proba, test_proba, fold_scores, cv_score, class_names = fit_catboost_cv(
    train_df=train_feat,
    test_df=test_feat,
    y=train[CFG.TARGET_COL].copy(),
    feature_cols=feature_cols,
    cat_features=cat_features,
)

print("class_names =", class_names)
print("fold_scores =", fold_scores)
print("cv_score =", cv_score)

## 9. Save artifacts

In [ ]:
np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba.npy", oof_proba)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba.npy", test_proba)

oof_pred_idx = oof_proba.argmax(axis=1)
oof_pred_label = pd.Series([class_names[i] for i in oof_pred_idx])

oof_df = pd.DataFrame({
    CFG.ID_COL: train[CFG.ID_COL],
    "y_true": train[CFG.TARGET_COL],
    "y_pred": oof_pred_label,
})
oof_df.to_csv(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_labels.csv", index=False)

sub = pd.DataFrame({
    CFG.ID_COL: test[CFG.ID_COL],
    CFG.TARGET_COL: [class_names[i] for i in test_proba.argmax(axis=1)]
})
sub.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

feature_columns_df = pd.DataFrame({
    "feature": feature_cols,
    "is_base_cat": [c in cat_features for c in feature_cols],
    "is_catnum": [c in catnum_cols for c in feature_cols],
})
feature_columns_df.to_csv(CFG.SAVE_DIR / "feature_columns.csv", index=False)

with open(CFG.SAVE_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "cv_score": cv_score,
            "fold_scores": fold_scores,
            "class_names": class_names,
            "n_features": len(feature_cols),
            "n_cat_features": len(cat_features),
            "n_catnum_cols": len(catnum_cols),
            "high_iv_top4": CFG.HIGH_IV_TOP4,
            "catnum_stats": CFG.CATNUM_STATS,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

## 10. Quick summary

In [ ]:
summary_df = pd.DataFrame({
    "item": [
        "cv_score",
        "n_features",
        "n_cat_features",
        "n_catnum_cols",
    ],
    "value": [
        cv_score,
        len(feature_cols),
        len(cat_features),
        len(catnum_cols),
    ]
})
display(summary_df)